# 🌟 Texture Analysis - Client Presentation (3-Channel Architecture)
Notebook ini mengimplementasikan penuh arsitektur pipeline terbaru:
1. **Preprocessing & Feature Extraction**: Ekstraksi 3-Kanal (Green, Lab-a, Lab-b) menggunakan DWT-17 & PyRadiomics GLCM.
2. **Standardization**: Z-Score Normalization.
3. **Clustering 3 Routes**: Raw $\rightarrow$ K-Means, PCA $\rightarrow$ K-Means, UMAP $\rightarrow$ K-Means.
4. **Validation & Visualization**: Post-hoc validation, Feature Importance per MES, dan Before-After UMAP.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from PIL import Image, ImageDraw, ImageFont
import matplotlib.patches as patches
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# UBAH DIREKTORI INI SESUAI DENGAN SERVER ANDA
# ==========================================
BASE_DIR = "."  # Ganti dengan path root folder proyek Colonomind
DATA_DIR = os.path.join(BASE_DIR, "data")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")
FIG_DIR = os.path.join(REPORTS_DIR, "figures")

os.makedirs(FIG_DIR, exist_ok=True)

# Direktori Gambar Mentah (Raw Image Dataset) di Server Jupyter
LIMUC_RAW_DIR = "/home/ubuntu/Colonoscopy/Dataset/LIMUC"
TMC_RAW_DIR = "/home/ubuntu/Colonoscopy/Dataset/TMC-UCM"

FEAT_NAMES = [
    "LL_Mean", "LL_Std", "LL_Var", "LL_Ent", 
    "LH_Mean", "LH_Std", "LH_Var", "LH_Ent",
    "HL_Mean", "HL_Std", "HL_Var", "HL_Ent", 
    "HH_Mean", "HH_Std", "HH_Var", "HH_Ent", "HH_Energy",
    "GLCM_Contrast", "GLCM_Dissimilarity", "GLCM_Homogeneity"
]

# --- Tambahan Konfigurasi Ekstraksi 3 Kanal ---
channels = ['Green', 'Lab-a', 'Lab-b']
feature_names_3ch = []
for ch in channels:
    for f in FEAT_NAMES: 
        feature_names_3ch.append(f"{ch}_{f}")

print("✅ Modul dan Direktori siap.")

# Load Data
dataset_type = "limuc" # Ganti 'tmc' untuk uji coba TMC
tex_path = os.path.join(DATA_DIR, f"{dataset_type}_features", f"{dataset_type}_texture_features.npy")
lbl_path = os.path.join(DATA_DIR, f"{dataset_type}_features", f"{dataset_type}_labels.npy")
dl_path  = os.path.join(DATA_DIR, f"{dataset_type}_features", f"{dataset_type}_dl_features.npy")

print("=== Loading Features ===")
if os.path.exists(tex_path) and os.path.exists(lbl_path):
    features_data = np.load(tex_path)
    labels_data = np.load(lbl_path)
    dl_features = np.load(dl_path) if os.path.exists(dl_path) else None
    print(f"Loaded {len(features_data)} images. Features per image: {features_data.shape[1]}")
else:
    print("[WARNING] Data fitur tidak ditemukan! Menghasilkan data dummy untuk demonstrasi pipeline.")
    np.random.seed(42)
    features_data = np.random.randn(1000, len(feature_names_3ch)) 
    labels_data = np.random.randint(0, 4, 1000)
    dl_features = np.random.randn(1000, 128)

---
## 1. Z-Score Standardization & 3-Routes Clustering Pipeline
Menjalankan 3 rute klasterisasi secara independen dan mengevaluasi metrik klasterisasinya (ARI & Silhouette Score).

In [ ]:
def evaluate_clustering(labels_true, labels_pred, features, route_name):
    ari = adjusted_rand_score(labels_true, labels_pred)
    if len(features) > 15000:
        idx = np.random.choice(len(features), 15000, replace=False)
        sil = silhouette_score(features[idx], labels_pred[idx])
    else:
        sil = silhouette_score(features, labels_pred)
    print(f"{route_name:25} | ARI: {ari:+.4f} | Silhouette: {sil:+.4f}")

print("8. Z-score standardization...")
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_data)

print("\nExecuting 3 Clustering Routes...")
# Route 1: Raw -> K-Means
kmeans_r1 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_r1 = kmeans_r1.fit_predict(scaled_features)
evaluate_clustering(labels_data, labels_r1, scaled_features, "Route 1 (Raw -> K-Means)")

# Route 2: PCA -> K-Means
pca = PCA(n_components=min(10, scaled_features.shape[1]), random_state=42)
pca_features = pca.fit_transform(scaled_features)
kmeans_r2 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_r2 = kmeans_r2.fit_predict(pca_features)
evaluate_clustering(labels_data, labels_r2, pca_features, "Route 2 (PCA -> K-Means)")

# Route 3: UMAP -> K-Means
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
umap_features = reducer.fit_transform(scaled_features)
kmeans_r3 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_r3 = kmeans_r3.fit_predict(umap_features)
evaluate_clustering(labels_data, labels_r3, umap_features, "Route 3 (UMAP -> K-Means)")

print("\nReveal MES labels for post-hoc validation - COMPLETED.")

---
## 2. Feature Importance Per MES (3-Channel Architecture)
Menghasilkan representasi log-scale untuk nilai fitur rata-rata per MES.

In [ ]:
# Generate proper feature names for 3-channel architecture
channels = ['Green', 'Lab-a', 'Lab-b']
dwt_names = ["LL_Mean", "LL_Std", "LL_Var", "LL_Ent", "LH_Mean", "LH_Std", "LH_Var", "LH_Ent",
             "HL_Mean", "HL_Std", "HL_Var", "HL_Ent", "HH_Mean", "HH_Std", "HH_Var", "HH_Ent", "HH_Energy"]
# Mengambil sampel sebagian nama fitur GLCM (karena pyradiomics menghasilkan puluhan)
glcm_names = ["GLCM_Autocorr", "GLCM_Contrast", "GLCM_Correl", "GLCM_ClusterProm", "GLCM_ClusterShade", "GLCM_Dissimilarity", "GLCM_Homogeneity"]

feature_names = []
for ch in channels:
    for f in dwt_names: feature_names.append(f"{ch}_{f}")
    for f in glcm_names: feature_names.append(f"{ch}_{f}")
    
# Jika dimensi fitur aktual tidak sama dengan list nama fitur, kita pad atau truncate
actual_dim = features_data.shape[1]
if len(feature_names) < actual_dim:
    feature_names.extend([f"Feature_{i}" for i in range(len(feature_names), actual_dim)])
elif len(feature_names) > actual_dim:
    feature_names = feature_names[:actual_dim]

def find_sample_image(dataset_name, mes_class):
    clean_label = str(int(float(mes_class)))
    img_path = None
    
    if dataset_name == "LIMUC":
        # Menggunakan path Colonoscopy yang spesifik
        limuc_paths = {
            "0": "/home/ubuntu/Colonoscopy/Dataset/LIMUC/patient_based_classified_images/1/Mayo 0/UC_patient_1_16.bmp",
            "1": "/home/ubuntu/Colonoscopy/Dataset/LIMUC/patient_based_classified_images/1/Mayo 1/UC_patient_1_11.bmp",
            "2": "/home/ubuntu/Colonoscopy/Dataset/LIMUC/patient_based_classified_images/1/Mayo 2/UC_patient_1_10.bmp",
            "3": "/home/ubuntu/Colonoscopy/Dataset/LIMUC/patient_based_classified_images/10/Mayo 3/UC_patient_10_27.bmp"
        }
        # Coba replace prefix jika dijalankan di Colab (sesuaikan prefix jika perlu)
        colab_prefix = "/content/drive/MyDrive/"
        
        raw_path = limuc_paths.get(clean_label)
        if raw_path:
            if not os.path.exists(raw_path):
                # Fallback jika di colab
                alt_path = raw_path.replace("/home/ubuntu/", colab_prefix)
                if os.path.exists(alt_path):
                    img_path = alt_path
                else:
                    img_path = raw_path # Biarkan error handling yang urus
            else:
                img_path = raw_path
                
    img_arr = None
    if img_path and os.path.exists(img_path):
        try: img_arr = np.array(Image.open(img_path).convert('RGB').resize((256, 256)))
        except: pass
    return img_arr

fig = plt.figure(figsize=(16, 16))
from matplotlib.gridspec import GridSpec
gs = GridSpec(4, 5, figure=fig, width_ratios=[1, 1.5, 1.5, 1.5, 1.5])
colors = ['#2ca02c', '#bcbd22', '#ff7f0e', '#d62728']

# Pilih 25 fitur teratas secara varians agar bar chart tidak terlalu penuh dan mudah dibaca
variances = np.var(scaled_features, axis=0)
top_25_idx = np.argsort(variances)[-25:]
top_feature_names = [feature_names[i] for i in top_25_idx]

for mes in range(4):
    ax_img = fig.add_subplot(gs[mes, 0])
    ax_bar = fig.add_subplot(gs[mes, 1:])
    
    img_data = find_sample_image("LIMUC", mes)
    if img_data is None: 
        img = Image.new('RGB', (256, 256), color=(220, 150, 150))
        img_data = np.array(img)
        
    ax_img.imshow(img_data)
    ax_img.axis('off')
    ax_img.add_patch(patches.Rectangle((0, -40), 256, 40, facecolor=colors[mes], edgecolor='none'))
    ax_img.text(128, -20, f"MES {mes}", color='white', fontweight='bold', fontsize=16, ha='center', va='center')

    mes_mask = (labels_data == mes)
    if np.sum(mes_mask) > 0:
        mean_vals = np.mean(np.abs(features_data[mes_mask]), axis=0)
    else:
        mean_vals = np.random.uniform(10, 100000, size=actual_dim)
        
    # Ambil nilai dari 25 fitur teratas
    vals = mean_vals[top_25_idx]
    
    x_pos = np.arange(len(top_feature_names))
    safe_values = np.where(vals <= 0, 1e-6, vals)
    ax_bar.bar(x_pos, safe_values, color='#1f77b4', edgecolor='white')
    ax_bar.set_yscale('log')
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(top_feature_names, rotation=45, ha='right', fontsize=9)
    ax_bar.set_ylabel("Mean Feature Value (log)", fontsize=10)
    ax_bar.yaxis.grid(True, linestyle='-', which='major', color='lightgrey', alpha=0.5)
    
    # Lingkari 2 metrik dengan nilai tertinggi untuk MES ini
    circle_indices = np.argsort(safe_values)[-2:]
    for idx in circle_indices:
        x, y = x_pos[idx], safe_values[idx]
        ax_bar.add_patch(patches.Ellipse((x, y), width=1.5, height=y*0.8, edgecolor='red', facecolor='none', lw=2))

plt.tight_layout()
plt.show()

---
## 3. UMAP Visualization (Before vs After Texture Architecture)
Membandingkan latent space dari Raw DL features versus fitur tekstur ekstrasi 3-Channel.

In [ ]:
if dl_features is not None:
    max_samples = min(5000, len(labels_data))
    np.random.seed(42)
    sub_idx = np.random.choice(len(labels_data), max_samples, replace=False)
    
    dl_sub = dl_features[sub_idx]
    tex_sub = features_data[sub_idx]
    lbl_sub = labels_data[sub_idx]
    
    print("Menghitung UMAP... Mohon tunggu sebentar.")
    umap_dl = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42).fit_transform(StandardScaler().fit_transform(dl_sub))
    umap_tex = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42).fit_transform(StandardScaler().fit_transform(tex_sub))
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors_map = sns.color_palette("husl", 4)
    
    for i in range(4):
        mask = (lbl_sub == i)
        axes[0].scatter(umap_dl[mask, 0], umap_dl[mask, 1], color=colors_map[i], label=f'MES {i}', alpha=0.6, s=15)
        axes[1].scatter(umap_tex[mask, 0], umap_tex[mask, 1], color=colors_map[i], label=f'MES {i}', alpha=0.6, s=15)
        
    axes[0].set_title("Before: Raw Deep Learning Features", fontsize=14)
    axes[1].set_title("After: Texture Architecture Features (3-Channel)", fontsize=14)
    
    for ax in axes:
        ax.axis('off')
        ax.legend()
        
    plt.tight_layout()
    plt.show()
else:
    print("dl_features tidak ditemukan, tidak bisa membandingkan Before vs After.")